# High-Score Object Distribution by Size

This notebook computes the percentage of high-score objects (top quartile by object score S_obj) that fall into each size category, and reports actual counts.

Size categories use equivalent square side length in pixels:
- very_small: < 20 px
- small_20_32: 20 to 32 px
- medium_gt_32: > 32 px

In [3]:
from pathlib import Path
import json
import numpy as np
from PIL import Image

# Update this path if you want a different score file.
SCORE_PATH = Path("/home/khanh/Projects/DifficultyAgri/.cache_result/no_trad_aug/minneapple/scoring/seed_123/score_results.json")

with SCORE_PATH.open("r", encoding="utf-8") as f:
    score_data = json.load(f)

image_difficulties = score_data.get("image_difficulties", [])
if not image_difficulties:
    raise ValueError(f"No image_difficulties found in {SCORE_PATH}")

shape_cache = {}

def get_image_size(image_path: str) -> tuple[int, int]:
    p = Path(image_path)
    if p not in shape_cache:
        with Image.open(p) as img:
            shape_cache[p] = img.size  # (width, height)
    return shape_cache[p]

s_obj = []
eq_side_px = []

for image_item in image_difficulties:
    for obj in image_item.get("objects_score", []):
        bbox = obj.get("bounding_box", {})
        w_norm = bbox.get("width")
        h_norm = bbox.get("height")
        score = obj.get("difficulty_score")
        image_path = obj.get("image_path")

        if w_norm is None or h_norm is None or score is None or image_path is None:
            continue

        img_w, img_h = get_image_size(image_path)
        area_px = float(w_norm) * float(h_norm) * float(img_w) * float(img_h)
        eq_side = float(np.sqrt(area_px))

        s_obj.append(float(score))
        eq_side_px.append(eq_side)

s_obj = np.asarray(s_obj, dtype=float)
eq_side_px = np.asarray(eq_side_px, dtype=float)

if len(s_obj) == 0:
    raise ValueError("No object-level scores found in objects_score.")

# Top quartile by S_obj
q75 = float(np.quantile(s_obj, 0.7))
high_mask = s_obj >= q75
high_count = int(high_mask.sum())

# Size categories
size_masks = {
    "very_small": eq_side_px < 20.0,
    "small_20_32": (eq_side_px >= 20.0) & (eq_side_px <= 32.0),
    "medium_gt_32": eq_side_px > 32.0,
}

print(f"Score file: {SCORE_PATH}")
print(f"Total objects: {len(s_obj)}")
print(f"Top-quartile threshold (S_obj, 75th percentile): {q75:.6f}")
print(f"High-score objects (top quartile): {high_count}")
print()
print("Distribution of top-quartile objects by size category")
print("category        high_count  total_in_cat  pct_of_high(%)  high_within_cat(%)")

for cat in ["very_small", "small_20_32", "medium_gt_32"]:
    cat_mask = size_masks[cat]
    n_cat = int(cat_mask.sum())
    n_high_cat = int((high_mask & cat_mask).sum())
    pct_of_high = (100.0 * n_high_cat / high_count) if high_count > 0 else float("nan")
    high_within_cat = (100.0 * n_high_cat / n_cat) if n_cat > 0 else float("nan")

    print(f"{cat:14s} {n_high_cat:10d} {n_cat:13d} {pct_of_high:14.2f} {high_within_cat:18.2f}")

Score file: /home/khanh/Projects/DifficultyAgri/.cache_result/no_trad_aug/minneapple/scoring/seed_123/score_results.json
Total objects: 22815
Top-quartile threshold (S_obj, 75th percentile): 0.230801
High-score objects (top quartile): 6845

Distribution of top-quartile objects by size category
category        high_count  total_in_cat  pct_of_high(%)  high_within_cat(%)
very_small           4323          5265          63.16              82.11
small_20_32          2316         11528          33.83              20.09
medium_gt_32          206          6022           3.01               3.42
